# Run 2: newer Qwen generation (Qwen 3, thinking off), same sample and settings
**Before running:** Settings -> *Accelerator* = **GPU T4 x2**, *Internet* = **On**, *Persistence* = **Files only**.
Your dataset must contain the **new** `run_experiment.py` (with the `--think` option), plus `prompts.py`, `analyze.py` and `sample.csv`.
Run the cells in order. If the session stops, run cells 1-4 and 6 again: `--resume` continues.

## 1. Record the hardware

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv > /kaggle/working/hardware_run2.txt
!cat /kaggle/working/hardware_run2.txt
!nproc

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
4


## 2. Copy scripts and sample

In [2]:
import glob, shutil, os
os.chdir('/kaggle/working')
found = glob.glob('/kaggle/input/**/sample.csv', recursive=True)
assert found, 'sample.csv not found - add your dataset with Add Input'
src = os.path.dirname(found[0])
for f in ['sample.csv', 'prompts.py', 'run_experiment.py', 'analyze.py']:
    shutil.copy(os.path.join(src, f), '/kaggle/working/')
assert '--think' in open('run_experiment.py').read(), 'OLD run_experiment.py: upload the new version to your dataset'
print('OK, copied from', src)

OK, copied from /kaggle/input/datasets/dovydasgan/prompt-sensitivity


## 3. Install and start Ollama

In [3]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, requests
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama_run2.log', 'w'),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); print('Ollama is running'); break
    except Exception:
        time.sleep(2)
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                           21.7%##################                                   54.7%#####                       71.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is running
ollama version is 0.40.17


## 4. Download the models
Qwen 3 at the closest sizes to your Qwen 2.5 run (7B/14B/32B), plus the newest Qwen (3.8, only available as 27B).
**Do not stop this cell while it downloads.** If a download fails with `unexpected EOF`, run the cell again (it resumes).

In [4]:
MODELS = ['qwen3:8b', 'qwen3:14b', 'qwen3:32b', 'qwen3.8:27b']
for m in MODELS:
    !ollama pull {m}
!ollama list

[download progress output removed to reduce file size]
NAME           ID              SIZE      MODIFIED      
qwen3.8:27b    aaee06c39dcf    17 GB     5 seconds ago    
qwen3:32b      030ee887880f    20 GB     2 minutes ago    
qwen3:14b      bdbd181c33f2    9.3 GB    5 minutes ago    
qwen3:8b       500a1f067a9f    5.2 GB    6 minutes ago    


## 5. Pilot (2 posts per model), thinking off
Check: `raw_answer` must be a plain label (no `<think>` text), and `ollama ps` should show 100% GPU for each model.

In [5]:
import pandas as pd
def tag(m): return m.replace(':', '_').replace('.', '')
for m in MODELS:
    for other in MODELS:
        subprocess.run(['ollama', 'stop', other], capture_output=True)
    !python run_experiment.py --models {m} --limit 2 --think off --output pilot_run2_{tag(m)}.csv --resume
    !ollama ps
for f in sorted(glob.glob('pilot_run2_*.csv')):
    d = pd.read_csv(f)
    print(f, '| mean seconds:', round(d.seconds.iloc[1:].mean(), 2),
          '| invalid:', (d.label == 'invalid').sum(),
          '| contains <think>:', d.raw_answer.astype(str).str.contains('think', case=False).sum())
    print(d[['variant', 'gold', 'raw_answer', 'label']].head(8).to_string(), '\n')

Running 24 prompts ...
  24/24
Done. Results in pilot_run2_qwen3_8b.csv, settings in pilot_run2_qwen3_8b_metadata.json
NAME        ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:8b    500a1f067a9f    5.3 GB    100% GPU     2048       llamacpp    4 minutes from now    
Running 24 prompts ...
  24/24
Done. Results in pilot_run2_qwen3_14b.csv, settings in pilot_run2_qwen3_14b_metadata.json
NAME         ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:14b    bdbd181c33f2    9.3 GB    100% GPU     2048       llamacpp    4 minutes from now    
Running 24 prompts ...
  24/24
Done. Results in pilot_run2_qwen3_32b.csv, settings in pilot_run2_qwen3_32b_metadata.json
NAME         ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:32b    030ee887880f    20 GB    100% GPU     2048       llamacpp    4 minutes from now    
Running 24 prompts ...
  24/24
Done. Results in pilot_run2_qwen3

## 6. Full run (3,600 prompts per model)
Each model gets the GPUs to itself; `ollama ps` is printed before each model (should say 100% GPU).

In [6]:
for m in MODELS:
    for other in MODELS:
        subprocess.run(['ollama', 'stop', other], capture_output=True)
    requests.post('http://localhost:11434/api/generate',
                  json={'model': m, 'prompt': '', 'think': False, 'options': {'num_ctx': 2048}},
                  timeout=900)
    print(f'\n===== {m} =====')
    !ollama ps
    !python run_experiment.py --models {m} --repeats 3 --think off --output results_run2_{tag(m)}.csv --resume


===== qwen3:8b =====
NAME        ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:8b    500a1f067a9f    5.3 GB    100% GPU     2048       llamacpp    4 minutes from now    
Running 3600 prompts ...
  25/3600
  50/3600
  75/3600
  100/3600
  125/3600
  150/3600
  175/3600
  200/3600
  225/3600
  250/3600
  275/3600
  300/3600
  325/3600
  350/3600
  375/3600
  400/3600
  425/3600
  450/3600
  475/3600
  500/3600
  525/3600
  550/3600
  575/3600
  600/3600
  625/3600
  650/3600
  675/3600
  700/3600
  725/3600
  750/3600
  775/3600
  800/3600
  825/3600
  850/3600
  875/3600
  900/3600
  925/3600
  950/3600
  975/3600
  1000/3600
  1025/3600
  1050/3600
  1075/3600
  1100/3600
  1125/3600
  1150/3600
  1175/3600
  1200/3600
  1225/3600
  1250/3600
  1275/3600
  1300/3600
  1325/3600
  1350/3600
  1375/3600
  1400/3600
  1425/3600
  1450/3600
  1475/3600
  1500/3600
  1525/3600
  1550/3600
  1575/3600
  1600/3600
  1625/3600
  1650/3600
  1675/3600


## 7. Combine and analyse

In [7]:
files = sorted(glob.glob('results_run2_*.csv'))
r2 = pd.concat([pd.read_csv(f) for f in files])
print(r2.groupby('model').size())
r2.to_csv('results_run2.csv', index=False)
!python analyze.py --input results_run2.csv

model
qwen3.8:27b    3600
qwen3:14b      3600
qwen3:32b      3600
qwen3:8b       3600
dtype: int64

Accuracy per wording (share correct):
variant      v1_plain  v2_formal  v3_casual  v4_reordered
model                                                    
qwen3.8:27b     0.757      0.760      0.720         0.757
qwen3:14b       0.693      0.696      0.682         0.733
qwen3:32b       0.773      0.760      0.690         0.782
qwen3:8b        0.689      0.712      0.663         0.733

Stability summary:
      model  repeat_consistency  wording_consistency  accuracy_min  accuracy_max  accuracy_spread  invalid_rate  mean_seconds
qwen3.8:27b               1.000                0.827         0.720         0.760            0.040         0.001          0.68
  qwen3:14b               0.998                0.840         0.682         0.733            0.051         0.003          0.19
  qwen3:32b               0.999                0.727         0.690         0.782            0.092         0.000     

## 8. Download
From the Output panel: `results_run2_*.csv`, `results_run2_*_metadata.json`, `hardware_run2.txt`, `summary_*.csv`. Then **stop the session**.